# 02 - Xử lý dữ liệu (Preprocessing)### Đề tài: Phân loại nấm ăn được hay có độcNotebook này áp dụng các quyết định đã chốt ở bước EDA (`01_eda.ipynb`):loại `veil-type`, giữ `?` của `stalk-root` làm 1 category riêng, mã hoá One-Hot,chia train/test có `stratify`, và xuất `schema.json` + `preprocessor.joblib`để dùng lại ở bước huấn luyện (03_train) và AI Service.**Cách chạy:** Runtime → Restart & Run all. Cần có `dataset.zip` (từ `ai-models/data/`)và `preprocess.py` (từ `ai-models/src/`) trong cùng thư mục làm việc trên Colab.

In [ ]:
!pip -q install scikit-learn pandas joblib

In [ ]:
import syssys.path.append(".")   # đảm bảo import được preprocess.py nếu để cùng thư mụcimport jsonimport pandas as pdfrom preprocess import (    load_raw_data, clean_data, get_feature_columns, build_preprocessor,    encode_target, build_schema, split, TARGET_COL, DROP_COLS, MISSING_TOKEN)

## 1. Đọc dữ liệu gốc từ dataset.zip

In [ ]:
df_raw = load_raw_data("dataset.zip")print("Dữ liệu gốc:", df_raw.shape)df_raw.head()

## 2. Làm sạch dữ liệu**Quyết định (đã chốt từ EDA — Hình 1.2 và 1.4):**- Loại cột `veil-type` — chỉ có 1 giá trị duy nhất trong toàn bộ 8.124 mẫu, không mang thông tin phân biệt.- Giữ nguyên ký hiệu `'?'` ở cột `stalk-root` (30,5% mẫu) — coi là **một mức phân loại riêng** thay vì xoá dòng hoặc điền mode, vì xoá 30% dữ liệu là quá nhiều và việc "không xác định được gốc cuống" cũng có thể mang thông tin phân biệt.

In [ ]:
df_clean = clean_data(df_raw)print("Sau khi loại", DROP_COLS, "->", df_clean.shape)assert "veil-type" not in df_clean.columnsprint("Giá trị duy nhất của stalk-root (vẫn giữ '?'):", sorted(df_clean["stalk-root"].unique()))

## 3. Mã hoá biến mục tiêu (target)`class`: `e` (edible) → 0, `p` (poisonous) → 1

In [ ]:
y, mapping = encode_target(df_clean)print("Bảng ánh xạ:", mapping)print(y.value_counts())

## 4. Chia tập Train / TestDùng `stratify=y` để giữ đúng tỷ lệ 2 lớp (~51,8% / 48,2%) ở cả 2 tập, tránh rò rỉ dữ liệubằng cách chia **trước khi** fit bất kỳ bộ mã hoá/chuẩn hoá nào.

In [ ]:
feature_cols = get_feature_columns(df_clean)print(f"Số thuộc tính đầu vào: {len(feature_cols)}")X_train, X_test, y_train, y_test = split(df_clean, feature_cols, y, test_size=0.2, random_state=42)print("Train:", X_train.shape, "| Test:", X_test.shape)print("Tỷ lệ nhãn train:\n", y_train.value_counts(normalize=True))print("Tỷ lệ nhãn test:\n", y_test.value_counts(normalize=True))

## 5. Mã hoá One-Hot EncodingToàn bộ 21 thuộc tính đều là categorical → dùng `OneHotEncoder(handle_unknown="ignore")`duy nhất trong một `ColumnTransformer`. `handle_unknown="ignore"` giúp AI Service không bị lỗinếu sau này gặp giá trị lạ chưa từng thấy lúc huấn luyện.**Lưu ý chống rò rỉ dữ liệu:** chỉ `fit` bộ mã hoá trên `X_train`, sau đó `transform` cả`X_train` và `X_test` — không bao giờ `fit` trên toàn bộ dữ liệu trước khi chia.

In [ ]:
preprocessor = build_preprocessor(feature_cols)X_train_enc = preprocessor.fit_transform(X_train)X_test_enc = preprocessor.transform(X_test)print("Số chiều sau One-Hot:", X_train_enc.shape[1], "cột (từ", len(feature_cols), "thuộc tính gốc)")print("Kích thước X_train sau mã hoá:", X_train_enc.shape)print("Kích thước X_test sau mã hoá:", X_test_enc.shape)

## 6. Sinh schema.json`schema.json` là "hợp đồng dữ liệu" dùng chung giữa notebook huấn luyện, Backend (validate input)và Frontend (sinh form nhập liệu) — đảm bảo tất cả đọc đúng cùng danh sách cột, kiểu, giá trị hợp lệ.

In [ ]:
schema = build_schema(df_clean, feature_cols, mapping)with open("schema.json", "w", encoding="utf-8") as f:    json.dump(schema, f, ensure_ascii=False, indent=2)print(json.dumps(schema, ensure_ascii=False, indent=2)[:800], "...")

## 7. Lưu Pipeline tiền xử lýLưu `preprocessor.joblib` để tái sử dụng ở bước huấn luyện (`03_train.ipynb`, ghép chung vớimodel đã chọn thành 1 Pipeline duy nhất) và ở AI Service sau này (Bước 5.1 — đóng gói model).

In [ ]:
import joblibjoblib.dump(preprocessor, "preprocessor.joblib")print("Đã lưu preprocessor.joblib và schema.json — copy 2 file này vào ai-models/models/ trong repo.")

## Tổng kết Bước 2 (chuyển sang Bước 3 — Huấn luyện mô hình)- Dữ liệu sạch: 8.124 dòng × 21 thuộc tính đầu vào (đã loại `veil-type`).- `stalk-root='?'` được giữ làm 1 category riêng, không xoá dòng.- Train: 6.499 mẫu | Test: 1.625 mẫu (tỷ lệ nhãn giữ nguyên ~51,8%/48,2% nhờ `stratify`).- Sau One-Hot Encoding: 116 chiều đầu vào cho mô hình.- Artifact đã xuất: `schema.json`, `preprocessor.joblib` (trong `ai-models/models/`).